# CARE-Fusion — Colab / Kaggle runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dtlong1979/care-fusion-sentiment-detection-vi/blob/main/notebooks/CARE_Fusion_Colab.ipynb)

Chạy thực nghiệm CARE-Fusion trên GPU. **Runtime → Change runtime type → GPU** (T4 free, hoặc A100/L4 nếu Colab Pro).

Thứ tự: clone → cài deps + Java → preprocess (A) → resources (B) → train (D) → evaluate (F–G).

In [ ]:
# 0) GPU check
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 1) Clone repo (đổi sang token/SSH nếu repo còn private)
%cd /content
!git clone https://github.com/dtlong1979/care-fusion-sentiment-detection-vi.git
%cd care-fusion-sentiment-detection-vi

In [ ]:
# 2) Cài Java (cho VnCoreNLP) + Python deps
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q -r requirements.txt
!pip install -q -e .   # make `care_fusion` importable (src layout)

In [ ]:
# 3) (Tuỳ chọn) Mount Drive để lưu checkpoint qua các session
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/care-fusion', exist_ok=True)

In [ ]:
# 4) Part A — tiền xử lý (tải VnCoreNLP lần đầu)
!python -m care_fusion.preprocess --config configs/default.yaml

In [ ]:
# 5) Part B — q_j + PMI graph (chỉ từ train)
!python -m care_fusion.resources --config configs/default.yaml --steps q,pmi

## Phase 2 — baseline OOF → weak labels (B2) → train CARE-Fusion

`--smoke` chạy nhanh để kiểm pipeline; bỏ `--smoke` để chạy thật (5 fold, đủ epoch, đủ seed).

In [ ]:
# 6) Baseline OOF p_text (5-fold) -> weak regime labels (B2)
!python -m care_fusion.baselines --config configs/default.yaml --emit-oof
!python -m care_fusion.resources --config configs/default.yaml --steps weak --ptext artifacts/p_text_oof.json

# 7) FULL experiment matrix: B0-B4 + CARE-Fusion + ablations, all seeds -> results.json
CKPT = "/content/drive/MyDrive/care-fusion/checkpoints"
!python -m care_fusion.experiments --config configs/default.yaml --out $CKPT

# 8) Evaluation F1-F4 + G (regime-stratified table, causal, faithfulness, McNemar/Wilcoxon/bootstrap)
import glob
ckpt = sorted(glob.glob(f"{CKPT}/CARE_full_seed*.pt"))[0]
!python -m care_fusion.evaluate --config configs/default.yaml --care-ckpt $ckpt --baseline B1_text --preds-dir $CKPT/preds